In [20]:
"""
CATE Explorer v2 — Premium Redesign
=====================================
Drop-in replacement for run_cate_explorer().
Same data pipeline, completely rebuilt frontend.

Usage:
    from cate_explorer_v2 import run_cate_explorer_v2
    run_cate_explorer_v2(df, renderer="browser")

Or run standalone (requires the original module for data loading):
    python cate_explorer_v2.py
"""
from __future__ import annotations

import json
import math
import re
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.simplefilter("ignore")
pd.options.mode.chained_assignment = None

# ── Import data helpers from original module ──────────────────────────────────
# If running standalone, we import from the original file.
# If used as a library, caller passes a ready DataFrame.

FILE_PATH = r"Data/covariates_modeling_uplift_models_2026-03-13.csv"
OUTPUT_DIR = Path("Output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DROP_INDICATORS = [
    "BNLX_ChurnP_SKUd_test_export.csv",
    "BNLX_ChurnP_SKUd_controle_export.csv",
    "BNLX_ChurnP_niks_test_export.csv",
    "BNLX_ChurnP_niks_controle_export.csv",
]

TREATMENT_CONVERTER = {
    "BNLX_ChurnP_10_test_export.csv":       "treatment_1",
    "BNLX_ChurnP_10_controle_export.csv":    "control_1",
    "BNLX_ChurnP_25_test_export.csv":        "treatment_2",
    "BNLX_ChurnP_25_controle_export.csv":    "control_2",
    "BNLX_ChurnP_5eu_test_export.csv":       "treatment_3",
    "BNLX_ChurnP_5eu_controle_export.csv":   "control_3",
    "BNLX_ChurnP_10eu_test_export.csv":      "treatment_4",
    "BNLX_ChurnP_10eu_controle_export.csv":  "control_4",
    "BNLX_ChurnP_250_test_export.csv":       "treatment_5",
    "BNLX_ChurnP_250_controle_export.csv":   "control_5",
    "BNLX_ChurnP_500_test_export.csv":       "treatment_6",
    "BNLX_ChurnP_500_controle_export.csv":   "control_6",
    "BNLX_ChurnP_SKUe_test_export.csv":      "treatment_7",
    "BNLX_ChurnP_SKUe_controle_export.csv":  "control_7",
    "All treatment":                          "treatment_8",
    "All control":                            "control_8",
}

CAT_COLS = ["has_rfl", "gender", "country_sk"]
NUM_COLS = [
    "recency",  "monetary_value", "aov_per_customer",
    "length_of_relationship", "online_sales", "retail_sales",
    "food_total", "vhms_total", "sports_total", "beauty_total",
]

#  "total_volume", "frequency",
N_BINS = 10

COL_UNITS: dict[str, dict[str, str]] = {
    "recency":                {"prefix": "", "suffix": "d",  "axis_suffix": " (Days)"},
    "length_of_relationship": {"prefix": "", "suffix": "d",  "axis_suffix": " (Days)"},
    "monetary_value":         {"prefix": "€", "suffix": "", "axis_suffix": " (€)"},
    "aov_per_customer":       {"prefix": "€", "suffix": "", "axis_suffix": " (€)"},
    "online_sales":           {"prefix": "€", "suffix": "", "axis_suffix": " (€)"},
    "retail_sales":           {"prefix": "€", "suffix": "", "axis_suffix": " (€)"},
    "food_total":             {"prefix": "€", "suffix": "", "axis_suffix": " (€)"},
    "vhms_total":             {"prefix": "€", "suffix": "", "axis_suffix": " (€)"},
    "sports_total":           {"prefix": "€", "suffix": "", "axis_suffix": " (€)"},
    "beauty_total":           {"prefix": "€", "suffix": "", "axis_suffix": " (€)"},
}

# ── Palette v2 — refined dark theme ──────────────────────────────────────────
BG        = "#101318"
SURFACE   = "#161b27"
SURFACE2  = "#1c2233"
SURFACE3  = "#242b3e"
BORDER    = "#333b58"
GRID      = "#1e2538"
TEXT      = "#edf1fa"
TEXT2     = "#c0c8de"
MUTED     = "#7580a0"
C_CTRL    = "#8b95a8"       # neutral slate — "baseline / no intervention"
C_TRTM    = "#34d399"       # vivid green  — "treatment effect"
C_POS     = "#3ecf8e"
C_NEG     = "#e86a6a"
ACCENT    = "#3ecf8e"
ACCENT_DIM = "#2a9464"


# ╔═══════════════════════════════════════════════════════════════════════════════╗
# ║  DATA LOADING (same as original)                                             ║
# ╚═══════════════════════════════════════════════════════════════════════════════╝

def coerce_metrics_to_numeric(df, cols):
    df = df.copy()
    df[cols] = df[cols].replace({",": ""}, regex=True).apply(pd.to_numeric, errors="coerce")
    return df

def load_data(path: str = FILE_PATH) -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df[~df["treatment_indicator"].isin(DROP_INDICATORS)]
    df["treatment"] = df["treatment_indicator"].map(TREATMENT_CONVERTER)
    df_overall = df.copy()
    df_overall["treatment"] = df_overall["treatment_indicator"].apply(
        lambda x: "treatment_8" if "test" in x else "control_8")
    df = pd.concat([df, df_overall], ignore_index=True)
    df["experiment_k"] = df["treatment"].str.extract(r"(\d+)$").astype(int)
    df["gender"] = df["gender"].fillna("no_gender")
    source_nums = [c for c in NUM_COLS if c in df.columns]
    df = coerce_metrics_to_numeric(df, source_nums)
    df["aov_per_customer"] = np.where(
        df["frequency"] > 0, (df["monetary_value"] / df["frequency"]).round(0), 0)
    df[NUM_COLS] = df[NUM_COLS].fillna(0).astype("int64")
    df[CAT_COLS] = df[CAT_COLS].astype("object")
    return df

# ╔═══════════════════════════════════════════════════════════════════════════════╗
# ║  HELPERS (same logic, cleaned up)                                            ║
# ╚═══════════════════════════════════════════════════════════════════════════════╝

_grp   = lambda s: s.apply(lambda x: "Treatment" if str(x).startswith("treatment") else "Control")
pretty = lambda c: c.replace("_", " ").title()

def _exp_labels() -> dict:
    return {
        1: "10% discount",
        2: "25% discount",
        3: "€5 voucher",
        4: "€10 voucher",
        5: "250 loyalty points",
        6: "500 loyalty points",
        7: "SKUe",
        8: "All incentives",
    }

EXP_LABELS = _exp_labels()
def exp_name(k: int) -> str:
    return EXP_LABELS.get(k, str(k))

def _fmt_bin(val: str, col: str) -> str:
    u = COL_UNITS.get(col)
    if not u or val == "= 0": return val
    p, s = u.get("prefix", ""), u.get("suffix", "")
    return f"{p}{val}{s}"

def _col_axis_label(col: str) -> str:
    base = col.replace("_", " ").title()
    u = COL_UNITS.get(col)
    return base + u.get("axis_suffix", "") if u else base


def _bin_numeric(data, col, q=N_BINS):
    zero_frac = (data[col] == 0).mean()
    use_hybrid = zero_frac > 0.25

    if use_hybrid:
        is_zero = data[col] == 0
        data.loc[is_zero, "bin"] = "= 0"
        nonzero = data.loc[~is_zero, col]
        if nonzero.nunique() < 2:
            med = str(int(nonzero.median()))
            data.loc[~is_zero, "bin"] = med
            bin_order = ["= 0", med]
            bin_ranges = {"= 0": (0, 0, 0, len(bin_order))}
            nz_min, nz_max = int(round(nonzero.min())), int(round(nonzero.max()))
            bin_ranges[med] = (nz_min, nz_max, 1, len(bin_order))
            return data, bin_order, bin_ranges
        try:
            nz_bins, intervals = pd.qcut(nonzero, q=q-1, labels=False, duplicates="drop", retbins=True)
        except Exception:
            nz_bins, intervals = pd.qcut(nonzero, q=3, labels=False, duplicates="drop", retbins=True)
        medians = data.loc[~is_zero].assign(_bi=nz_bins).groupby("_bi")[col].median().round(0).astype(int).astype(str)
        data.loc[~is_zero, "bin"] = nz_bins.map(medians).values
        bin_order = ["= 0"] + medians.tolist()
        n_total = len(bin_order)
        bin_ranges = {"= 0": (0, 0, 0, n_total)}
        for idx in range(len(intervals) - 1):
            mk = medians.get(idx)
            if mk is not None:
                lo, hi = int(round(intervals[idx])), int(round(intervals[idx + 1]))
                bin_ranges[mk] = (lo, hi, idx + 1, n_total)
    else:
        try:
            data["bin_idx"], intervals = pd.qcut(data[col], q=q, labels=False, duplicates="drop", retbins=True)
        except Exception:
            return None, None, None
        medians = data.groupby("bin_idx")[col].median().round(0).astype(int).astype(str)
        data["bin"] = data["bin_idx"].map(medians)
        bin_order = medians.tolist()
        n_total = len(bin_order)
        bin_ranges = {}
        for idx in range(len(intervals) - 1):
            mk = medians.get(idx)
            if mk is not None:
                bin_ranges[mk] = (int(round(intervals[idx])), int(round(intervals[idx+1])), idx, n_total)

    return data, bin_order, bin_ranges


def _bin_label(bv, bin_ranges, is_cat):
    if is_cat: return str(bv)
    if str(bv) == "= 0": return "= 0"
    info = bin_ranges.get(str(bv))
    if info is None: return str(bv)
    lo, hi, idx, n = info
    if idx == 0: return f"< {hi:,}"
    if idx == n - 1: return f"> {lo:,}"
    return f"{lo:,} – {hi:,}"


def _pivot_bins(data, col, bin_order):
    agg = data.groupby(["bin", "group"])["reactivated"].agg(["sum", "count"]).rename(
        columns={"sum": "cb", "count": "n"}).reset_index()
    agg["rr"] = agg["cb"] / agg["n"]
    piv = agg.pivot(index="bin", columns="group", values="rr").reset_index()
    piv.columns.name = None
    cnt = agg.pivot(index="bin", columns="group", values="n").reset_index()
    cnt.columns.name = None
    for g in ["Control", "Treatment"]:
        if g not in piv.columns: piv[g] = np.nan
        if g not in cnt.columns: cnt[g] = 0
    piv["uplift"] = piv["Treatment"] - piv["Control"]
    piv["ctrl_n"] = cnt["Control"].fillna(0).astype(int).values
    piv["trt_n"]  = cnt["Treatment"].fillna(0).astype(int).values
    piv = piv.rename(columns={"Control": "ctrl_rr", "Treatment": "trt_rr"})
    piv["bin"] = piv["bin"].astype(str)
    order = {str(b): i for i, b in enumerate(bin_order)}
    piv["_o"] = piv["bin"].map(order).fillna(999)
    return piv.sort_values("_o").drop(columns="_o").reset_index(drop=True)


def compute_table(df, col, is_cat):
    data = df[["treatment", "reactivated", col]].copy().dropna(subset=[col, "reactivated"])
    data["group"] = _grp(data["treatment"])
    if is_cat:
        data["bin"] = data[col].astype(str)
        bin_order = sorted(data["bin"].unique())
    else:
        data, bin_order, _ = _bin_numeric(data, col)
        if bin_order is None: return None
    return _pivot_bins(data, col, bin_order)


# ╔═══════════════════════════════════════════════════════════════════════════════╗
# ║  V2 DASHBOARD BUILDER                                                       ║
# ╚═══════════════════════════════════════════════════════════════════════════════╝

def run_cate_explorer_v2(df: pd.DataFrame, renderer: str = "browser"):
    """Build and launch the redesigned CATE Explorer dashboard."""

    experiments = sorted(df["experiment_k"].dropna().unique().tolist())
    avail_num = [c for c in NUM_COLS if c in df.columns]
    avail_cat = [c for c in CAT_COLS if c in df.columns]
    all_cols  = avail_num + avail_cat
    col_is_cat = {c: (c in avail_cat) for c in all_cols}

    seg_options = [("All customers", None, None)]
    for cc in avail_cat:
        for val in sorted(df[cc].dropna().unique()):
            seg_options.append((f"{pretty(cc)}: {val}", cc, val))

    # ── Precompute all tables ─────────────────────────────────────────────────
    tables = {}
    for si, (_, sc, sv) in enumerate(seg_options):
        ds = df if sc is None else df[df[sc].astype(str) == str(sv)]
        for k in experiments:
            dk = ds[ds["experiment_k"] == k]
            if dk.empty: continue
            for col in all_cols:
                tbl = compute_table(dk, col, col_is_cat[col])
                if tbl is not None and not tbl.empty:
                    tables[(si, k, col)] = tbl

    if not tables:
        print("No data to plot."); return

    # ── Precompute experiment-level KPIs ──────────────────────────────────────
    exp_kpis = {}
    for si, (_, sc, sv) in enumerate(seg_options):
        ds = df if sc is None else df[df[sc].astype(str) == str(sv)]
        for k in experiments:
            dk = ds[ds["experiment_k"] == k]
            if dk.empty: continue
            dk_g = dk.copy()
            dk_g["_g"] = _grp(dk_g["treatment"])
            g = dk_g.groupby("_g")["reactivated"].agg(["mean", "count"])
            cr = g.loc["Control", "mean"] if "Control" in g.index else 0
            tr = g.loc["Treatment", "mean"] if "Treatment" in g.index else 0
            nc = int(g.loc["Control", "count"]) if "Control" in g.index else 0
            nt = int(g.loc["Treatment", "count"]) if "Treatment" in g.index else 0
            exp_kpis[f"{si},{k}"] = {
                "ctrl_rr": round(cr, 6), "trt_rr": round(tr, 6),
                "uplift": round(tr - cr, 6), "n_ctrl": nc, "n_trt": nt,
            }

    # ── Build serialisable table data ─────────────────────────────────────────
    combo_order = [(si, k, c) for si in range(len(seg_options))
                   for k in experiments for c in all_cols if (si, k, c) in tables]
    valid_exps  = sorted({k for _, k, _ in combo_order})
    valid_cols  = list(dict.fromkeys(c for _, _, c in combo_order))
    ei_map = {k: i for i, k in enumerate(valid_exps)}
    ci_map = {c: i for i, c in enumerate(valid_cols)}

    combo_matrix = [[[None]*len(valid_cols) for _ in range(len(valid_exps))]
                    for _ in range(len(seg_options))]

    # Serialise each table as JSON-ready dicts
    all_table_data = []
    for ti, (si, k, col) in enumerate(combo_order):
        tbl = tables[(si, k, col)]
        combo_matrix[si][ei_map[k]][ci_map[col]] = ti

        bins_formatted = [_fmt_bin(b, col) for b in tbl["bin"].tolist()]
        uplift_list = [round(v, 6) if pd.notna(v) else None for v in tbl["uplift"]]
        ctrl_rr_list = [round(v, 6) if pd.notna(v) else None for v in tbl["ctrl_rr"]]
        trt_rr_list = [round(v, 6) if pd.notna(v) else None for v in tbl["trt_rr"]]
        ctrl_n_list = [int(v) for v in tbl["ctrl_n"]]
        trt_n_list = [int(v) for v in tbl["trt_n"]]

        # Find best bin (highest uplift with min sample)
        best_idx = None
        best_uplift = -np.inf
        for j, u in enumerate(uplift_list):
            if u is not None and u > best_uplift and ctrl_n_list[j] >= 20 and trt_n_list[j] >= 20:
                best_uplift = u
                best_idx = j

        all_table_data.append({
            "bins": bins_formatted,
            "ctrl_rr": ctrl_rr_list,
            "trt_rr": trt_rr_list,
            "uplift": uplift_list,
            "ctrl_n": ctrl_n_list,
            "trt_n": trt_n_list,
            "best_idx": best_idx,
            "col": col,
            "exp_k": k,
        })

    # ── Build the JS data payload ─────────────────────────────────────────────
    js_payload = {
        "comboMatrix": combo_matrix,
        "tables": all_table_data,
        "expKpis": exp_kpis,
        "expLabels": [exp_name(k) for k in valid_exps],
        "expKeys": valid_exps,
        "colLabels": [pretty(c) for c in valid_cols],
        "colKeys": valid_cols,
        "colAxisLabels": [_col_axis_label(c) for c in valid_cols],
        "segLabels": [s[0] for s in seg_options],
        "nExp": len(valid_exps),
        "nCol": len(valid_cols),
        "nSeg": len(seg_options),
    }

    html = _build_html(js_payload)

    out = OUTPUT_DIR / "cate_explorer_v2.html"
    out.write_text(html, encoding="utf-8")
    print(f"  → {out}")

    if renderer == "browser":
        import tempfile, webbrowser
        tmp = tempfile.NamedTemporaryFile(suffix=".html", delete=False, mode="w", encoding="utf-8")
        tmp.write(html); tmp.close()
        webbrowser.open("file://" + tmp.name)
    else:
        from IPython.display import display, HTML as IHTML
        display(IHTML(html))


def _build_html(payload: dict) -> str:
    """Generate the full self-contained HTML dashboard."""
    data_json = json.dumps(payload)

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>CATE Explorer — Winback Experiment Dashboard</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=DM+Sans:ital,opsz,wght@0,9..40,300;0,9..40,400;0,9..40,500;0,9..40,600;0,9..40,700;1,9..40,400&family=Instrument+Serif:ital@0;1&display=swap" rel="stylesheet">
<style>
/* ── Reset & Base ─────────────────────────────────────────────────────────── */
*, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}

:root {{
  --bg:       {BG};
  --surface:  {SURFACE};
  --surface2: {SURFACE2};
  --surface3: {SURFACE3};
  --border:   {BORDER};
  --grid:     {GRID};
  --text:     {TEXT};
  --text2:    {TEXT2};
  --muted:    {MUTED};
  --ctrl:     {C_CTRL};
  --trtm:     {C_TRTM};
  --pos:      {C_POS};
  --neg:      {C_NEG};
  --accent:   {ACCENT};
  --accent-dim: {ACCENT_DIM};
  --font:     'DM Sans', -apple-system, BlinkMacSystemFont, sans-serif;
  --serif:    'Instrument Serif', Georgia, serif;
}}

body {{
  background: var(--bg);
  color: var(--text);
  font-family: var(--font);
  line-height: 1.5;
  -webkit-font-smoothing: antialiased;
  overflow-x: hidden;
}}

/* ── Grain overlay ────────────────────────────────────────────────────────── */
body::before {{
  content: '';
  position: fixed;
  inset: 0;
  pointer-events: none;
  z-index: 9999;
  opacity: 0.025;
  background-image: url("data:image/svg+xml,%3Csvg viewBox='0 0 256 256' xmlns='http://www.w3.org/2000/svg'%3E%3Cfilter id='noise'%3E%3CfeTurbulence type='fractalNoise' baseFrequency='0.9' numOctaves='4' stitchTiles='stitch'/%3E%3C/filter%3E%3Crect width='100%25' height='100%25' filter='url(%23noise)'/%3E%3C/svg%3E");
  background-size: 256px;
}}

/* ── Layout shell ─────────────────────────────────────────────────────────── */
.dashboard {{
  max-width: 1400px;
  margin: 0 auto;
  padding: 28px 32px 48px;
}}

/* ── Header ───────────────────────────────────────────────────────────────── */
.header {{
  display: flex;
  align-items: baseline;
  gap: 16px;
  margin-bottom: 6px;
}}
.header h1 {{
  font-family: var(--serif);
  font-size: 32px;
  font-weight: 400;
  color: var(--text);
  letter-spacing: -0.5px;
}}
.header .subtitle {{
  font-size: 13px;
  color: var(--text);
  font-weight: 700;
}}

/* ── Pill selectors ───────────────────────────────────────────────────────── */
.pill-row {{
  display: flex;
  align-items: center;
  gap: 8px;
  margin-bottom: 10px;
  flex-wrap: wrap;
}}
.pill-row .label {{
  font-size: 11px;
  font-weight: 700;
  text-transform: uppercase;
  letter-spacing: 0.8px;
  color: var(--text);
  margin-right: 4px;
  white-space: nowrap;
}}
.pill {{
  padding: 5px 12px;
  border-radius: 20px;
  font-size: 12.5px;
  font-weight: 500;
  font-family: var(--font);
  color: var(--text2);
  background: transparent;
  border: 1px solid var(--border);
  cursor: pointer;
  transition: all 0.2s ease;
  white-space: nowrap;
  user-select: none;
}}
.pill:hover {{
  background: var(--surface3);
  border-color: var(--muted);
  color: var(--text);
}}
.pill.active {{
  background: var(--accent);
  border-color: var(--accent);
  color: var(--bg);
  font-weight: 600;
  box-shadow: 0 0 16px rgba(62, 207, 142, 0.2);
}}

/* ── Segment dropdown ─────────────────────────────────────────────────────── */
.seg-wrap {{
  position: relative;
  display: inline-flex;
  align-items: center;
}}
.seg-btn {{
  padding: 6px 14px;
  border-radius: 20px;
  font-size: 12px;
  font-weight: 500;
  font-family: var(--font);
  color: var(--muted);
  background: transparent;
  border: 1px dashed var(--border);
  cursor: pointer;
  transition: all 0.2s ease;
}}
.seg-btn:hover {{ border-color: var(--muted); color: var(--text2); }}
.seg-btn.active-seg {{
  border-style: solid;
  border-color: var(--accent-dim);
  color: var(--accent);
}}
.seg-dropdown {{
  display: none;
  position: absolute;
  top: calc(100% + 6px);
  left: 0;
  z-index: 100;
  background: var(--surface2);
  border: 1px solid var(--border);
  border-radius: 12px;
  min-width: 240px;
  max-height: 360px;
  overflow-y: auto;
  box-shadow: 0 12px 48px rgba(0,0,0,0.5);
  padding: 6px;
}}
.seg-dropdown.open {{ display: block; }}
.seg-item {{
  padding: 8px 14px;
  font-size: 13px;
  cursor: pointer;
  border-radius: 8px;
  transition: background 0.15s;
  color: var(--text2);
}}
.seg-item:hover {{ background: var(--surface3); }}
.seg-item.active {{ background: var(--border); color: var(--accent); font-weight: 600; }}
.seg-divider {{ height: 1px; background: var(--border); margin: 4px 8px; }}

/* ── KPI strip ────────────────────────────────────────────────────────────── */
.kpi-strip {{
  display: grid;
  grid-template-columns: repeat(4, 1fr);
  gap: 12px;
  margin: 16px 0;
}}
.kpi-card {{
  background: var(--surface);
  border: 1px solid var(--border);
  border-radius: 12px;
  padding: 16px 20px;
  position: relative;
  overflow: hidden;
  transition: border-color 0.3s;
}}
.kpi-card::before {{
  content: '';
  position: absolute;
  inset: 0;
  background: linear-gradient(135deg, rgba(62,207,142,0.03) 0%, transparent 60%);
  pointer-events: none;
}}
.kpi-card .kpi-label {{
  font-size: 11px;
  font-weight: 700;
  text-transform: uppercase;
  letter-spacing: 0.7px;
  color: var(--text);
  margin-bottom: 6px;
}}
.kpi-card .kpi-value {{
  font-family: var(--font);
  font-size: 26px;
  font-weight: 700;
  letter-spacing: -0.5px;
}}
.kpi-card .kpi-sub {{
  font-size: 12px;
  color: var(--text2);
  margin-top: 4px;
}}
.kpi-pos {{ color: var(--pos); }}
.kpi-neg {{ color: var(--neg); }}
.kpi-neutral {{ color: var(--text); }}

/* ── Insight sentence ─────────────────────────────────────────────────────── */
.insight-bar {{
  display: flex;
  align-items: center;
  gap: 12px;
  padding: 12px 20px;
  background: var(--surface);
  border: 1px solid var(--border);
  border-radius: 12px;
  margin-bottom: 16px;
  min-height: 48px;
}}
.insight-bar .bulb {{
  flex-shrink: 0;
  width: 28px;
  height: 28px;
  border-radius: 50%;
  background: linear-gradient(135deg, var(--accent-dim), var(--accent));
  display: flex;
  align-items: center;
  justify-content: center;
  font-size: 14px;
}}
.insight-bar .insight-text {{
  font-size: 13.5px;
  color: var(--text2);
  line-height: 1.5;
  flex: 1;
}}
.insight-bar .insight-text strong {{
  color: var(--text);
  font-weight: 600;
}}
.insight-bar .insight-text .up-val {{
  color: var(--pos);
  font-weight: 700;
}}
.insight-bar .insight-text .down-val {{
  color: var(--neg);
  font-weight: 700;
}}
.copy-btn {{
  flex-shrink: 0;
  padding: 6px 14px;
  border-radius: 8px;
  font-size: 11px;
  font-weight: 600;
  text-transform: uppercase;
  letter-spacing: 0.5px;
  background: var(--surface3);
  border: 1px solid var(--border);
  color: var(--text2);
  cursor: pointer;
  transition: all 0.2s;
  font-family: var(--font);
}}
.copy-btn:hover {{ background: var(--border); color: var(--text); }}
.copy-btn.copied {{ background: var(--accent-dim); color: var(--bg); border-color: var(--accent-dim); }}

/* ── Chart container ──────────────────────────────────────────────────────── */
.chart-section {{
  position: relative;
}}
.chart-title {{
  font-family: var(--serif);
  font-size: 22px;
  font-weight: 400;
  color: var(--text);
  margin-bottom: 4px;
  font-style: italic;
}}
.chart-title .exp-tag {{
  font-family: var(--font);
  font-style: normal;
  font-size: 12px;
  font-weight: 600;
  color: var(--accent);
  background: rgba(62,207,142,0.1);
  padding: 3px 10px;
  border-radius: 12px;
  margin-left: 12px;
  vertical-align: middle;
  letter-spacing: 0.3px;
}}
/* ── Best bin annotation pulse ────────────────────────────────────────────── */
@keyframes pulse-glow {{
  0%, 100% {{ opacity: 0.6; }}
  50% {{ opacity: 1; }}
}}

/* ── Responsive ───────────────────────────────────────────────────────────── */
@media (max-width: 900px) {{
  .kpi-strip {{ grid-template-columns: repeat(2, 1fr); }}
  .dashboard {{ padding: 16px; }}
}}

/* ── Plotly overrides ─────────────────────────────────────────────────────── */
.plotly .modebar {{ opacity: 0 !important; transition: opacity 0.3s !important; }}
.plotly:hover .modebar {{ opacity: 0.5 !important; }}


</style>
</head>
<body>

<div class="dashboard">
  <!-- Header -->
  <div class="header">
    <h1>CATE Explorer</h1>
    <span class="subtitle">Conditional Average Treatment Effect · Winback Experiments</span>
  </div>

  <!-- Experiment pills -->
  <div class="pill-row" id="exp-pills">
    <span class="label">Experiment</span>
  </div>

  <!-- Variable pills -->
  <div class="pill-row" id="cov-pills">
    <span class="label">Variable</span>
  </div>

  <!-- Segment filter row -->
  <div class="pill-row" id="seg-row">
    <span class="label">Segment</span>
  </div>

  <!-- KPI strip -->
  <div class="kpi-strip" id="kpi-strip"></div>

  <!-- Insight bar -->
  <div class="insight-bar" id="insight-bar">
    <div class="bulb">💡</div>
    <div class="insight-text" id="insight-text">Loading…</div>
    <button class="copy-btn" id="copy-btn" onclick="copyInsight()">Copy</button>
  </div>

  <!-- Chart title -->
  <div class="chart-section">
    <div class="chart-title" id="chart-title"></div>
    <div id="chart-uplift" style="border-radius:16px 16px 0 0;overflow:hidden;border:1px solid var(--border);border-bottom:none;background:var(--surface);"></div>
    <div id="chart-rr" style="border-radius:0 0 16px 16px;overflow:hidden;border:1px solid var(--border);border-top:none;background:var(--surface);"></div>
  </div>
  </div>
</div>



<script>
(function() {{
  "use strict";

  const D = {data_json};

  // ── State ──────────────────────────────────────────────────────────────────
  let aE = 0, aC = 0, aS = 0;
  let currentInsightText = "";

  const pct  = v => v == null ? "—" : (v * 100).toFixed(2) + "%";
  const spct = v => v == null ? "—" : (v >= 0 ? "+" : "") + (v * 100).toFixed(2) + "pp";

  // ── Build experiment pills ─────────────────────────────────────────────────
  const expRow = document.getElementById("exp-pills");
  D.expLabels.forEach((label, i) => {{
    const btn = document.createElement("button");
    btn.className = "pill" + (i === 0 ? " active" : "");
    btn.textContent = label;
    btn.dataset.idx = i;
    btn.addEventListener("click", () => {{ aE = i; apply(); }});
    expRow.appendChild(btn);
  }});

  // ── Build covariate pills + segment dropdown ───────────────────────────────
  const covRow = document.getElementById("cov-pills");
  D.colLabels.forEach((label, i) => {{
    const btn = document.createElement("button");
    btn.className = "pill" + (i === 0 ? " active" : "");
    btn.textContent = label;
    btn.dataset.idx = i;
    btn.addEventListener("click", () => {{ aC = i; apply(); }});
    covRow.appendChild(btn);
  }});

  // Segment dropdown on its own row
  if (D.nSeg > 1) {{
    const segRow = document.getElementById("seg-row");
    const wrap = document.createElement("div");
    wrap.className = "seg-wrap";

    const btn = document.createElement("button");
    btn.className = "seg-btn";
    btn.id = "seg-btn";
    btn.textContent = "All customers ▾";
    btn.addEventListener("click", e => {{
      e.stopPropagation();
      document.getElementById("seg-dd").classList.toggle("open");
    }});
    wrap.appendChild(btn);

    const dd = document.createElement("div");
    dd.className = "seg-dropdown";
    dd.id = "seg-dd";
    D.segLabels.forEach((label, i) => {{
      if (i === 1) {{
        const div = document.createElement("div");
        div.className = "seg-divider";
        dd.appendChild(div);
      }}
      const item = document.createElement("div");
      item.className = "seg-item" + (i === 0 ? " active" : "");
      item.textContent = label;
      item.dataset.idx = i;
      item.addEventListener("click", e => {{
        e.stopPropagation();
        aS = i;
        dd.classList.remove("open");
        apply();
      }});
      dd.appendChild(item);
    }});
    wrap.appendChild(dd);
    segRow.appendChild(wrap);

    document.addEventListener("click", () => dd.classList.remove("open"));
  }}

  // ── Shared axis / layout constants for perfect alignment ───────────────────
  const SHARED_MARGIN_L = 80;
  const SHARED_MARGIN_R = 32;
  const SHARED_BARGAP   = 0.25;

  // ── Main apply function ────────────────────────────────────────────────────
  function apply() {{
    let ti = D.comboMatrix[aS]?.[aE]?.[aC];
    if (ti == null) {{
      ti = D.comboMatrix[0]?.[aE]?.[aC];
      if (ti == null) return;
      aS = 0;
    }}

    updatePills();
    updateKPIs();
    renderChart(ti);
    updateInsight(ti);
    updateChartTitle();
  }}

  function updatePills() {{
    document.querySelectorAll("#exp-pills .pill").forEach((el, i) => {{
      el.classList.toggle("active", i === aE);
    }});
    document.querySelectorAll("#cov-pills .pill").forEach((el, i) => {{
      el.classList.toggle("active", i === aC);
    }});
    const segBtn = document.getElementById("seg-btn");
    if (segBtn) {{
      segBtn.textContent = D.segLabels[aS] + " ▾";
      segBtn.classList.toggle("active-seg", aS > 0);
    }}
    document.querySelectorAll(".seg-item").forEach((el, i) => {{
      el.classList.toggle("active", i === aS);
    }});
  }}

  function updateKPIs() {{
    const key = aS + "," + D.expKeys[aE];
    const kpi = D.expKpis[key] || {{}};
    const strip = document.getElementById("kpi-strip");

    const uplift = kpi.uplift || 0;
    const upliftClass = uplift >= 0 ? "kpi-pos" : "kpi-neg";

    strip.innerHTML = `
      <div class="kpi-card">
        <div class="kpi-label">Overall Uplift</div>
        <div class="kpi-value ${{upliftClass}}">${{spct(uplift)}}</div>
        <div class="kpi-sub">Treatment – Control</div>
      </div>
      <div class="kpi-card">
        <div class="kpi-label">Control Rate</div>
        <div class="kpi-value kpi-neutral">${{pct(kpi.ctrl_rr)}}</div>
        <div class="kpi-sub">n = ${{(kpi.n_ctrl || 0).toLocaleString()}}</div>
      </div>
      <div class="kpi-card">
        <div class="kpi-label">Treatment Rate</div>
        <div class="kpi-value kpi-neutral">${{pct(kpi.trt_rr)}}</div>
        <div class="kpi-sub">n = ${{(kpi.n_trt || 0).toLocaleString()}}</div>
      </div>
      <div class="kpi-card">
        <div class="kpi-label">Total Customers</div>
        <div class="kpi-value kpi-neutral">${{((kpi.n_ctrl || 0) + (kpi.n_trt || 0)).toLocaleString()}}</div>
        <div class="kpi-sub">Control + Treatment</div>
      </div>
    `;
  }}

  function updateChartTitle() {{
    const el = document.getElementById("chart-title");
    el.innerHTML = `${{D.colLabels[aC]}} <span class="exp-tag">${{D.expLabels[aE]}}</span>`;
  }}

  // ── Chart rendering (two independent Plotly charts, locked alignment) ──────
  let gdUp = null, gdRr = null;

  function renderChart(ti) {{
    const t = D.tables[ti];
    if (!t) return;

    const bins = t.bins;

    // ── A/B testing colour conventions ───────────────────────────────────────
    // Control  = neutral grey  (baseline, no intervention)
    // Treatment = green        (the effect we care about)
    const CTRL_FILL  = "rgba(139,149,168,0.50)";   // slate grey
    const CTRL_LINE  = "rgba(139,149,168,0.75)";
    const TRT_FILL   = "rgba(52,211,153,0.65)";    // green
    const TRT_LINE   = "rgba(52,211,153,0.90)";

    // Uplift bar colors — green for positive, red for negative
    const upliftColors = t.uplift.map(v =>
      v == null ? "rgba(90,100,136,0.3)" : v >= 0
        ? `rgba(52,211,153,${{0.45 + Math.min(Math.abs(v) * 18, 0.45)}})`
        : `rgba(232,106,106,${{0.45 + Math.min(Math.abs(v) * 18, 0.45)}})`
    );
    const upliftBorders = t.uplift.map(v =>
      v == null ? "rgba(90,100,136,0.15)" : v >= 0
        ? "rgba(52,211,153,0.85)"
        : "rgba(232,106,106,0.85)"
    );

    // Best bin annotation
    const bestIdx = t.best_idx;
    const annotationsUp = [];
    if (bestIdx != null && t.uplift[bestIdx] != null) {{
      annotationsUp.push({{
        x: bins[bestIdx],
        y: t.uplift[bestIdx],
        xref: "x", yref: "y",
        text: `<b>${{spct(t.uplift[bestIdx])}}</b>`,
        showarrow: true, arrowhead: 0, arrowwidth: 1.5,
        arrowcolor: "{ACCENT}", ax: 0, ay: -32,
        font: {{ size: 13, color: "{ACCENT}", family: "DM Sans" }},
        bgcolor: "rgba(62,207,142,0.1)",
        bordercolor: "{ACCENT}", borderwidth: 1, borderpad: 5,
      }});
    }}

    // Hover text
    const hoverUplift = bins.map((b, i) => {{
      return `<b>${{b}}</b><br>Uplift: <b>${{pct(t.uplift[i])}}</b><br>n ctrl: ${{t.ctrl_n[i].toLocaleString()}}  ·  n trt: ${{t.trt_n[i].toLocaleString()}}`;
    }});
    const hoverCtrl = bins.map((b, i) => {{
      return `<b>${{b}}</b><br>Control RR: <b>${{pct(t.ctrl_rr[i])}}</b><br>n = ${{t.ctrl_n[i].toLocaleString()}}`;
    }});
    const hoverTrt = bins.map((b, i) => {{
      return `<b>${{b}}</b><br>Treatment RR: <b>${{pct(t.trt_rr[i])}}</b><br>n = ${{t.trt_n[i].toLocaleString()}}`;
    }});

    const hlStyle = {{
      bgcolor: "{SURFACE2}",
      bordercolor: "{BORDER}",
      font: {{ size: 12, color: "{TEXT}", family: "DM Sans, sans-serif" }},
    }};

    const configPlotly = {{
      displayModeBar: true,
      modeBarButtonsToRemove: ["select2d", "lasso2d", "autoScale2d"],
      responsive: true,
    }};

    // ── Shared x-axis template (ensures identical category axis) ─────────────
    const sharedXAxis = {{
      type: "category",
      showgrid: false,
      linecolor: "{BORDER}",
      tickfont: {{ size: 11, color: "{TEXT}", family: "DM Sans", weight: "bold" }},
      tickcolor: "{BORDER}",
      fixedrange: true,
      categoryorder: "array",
      categoryarray: bins,
    }};

    // ── UPLIFT CHART (top) ───────────────────────────────────────────────────
    const upliftTrace = {{
      x: bins, y: t.uplift, type: "bar",
      marker: {{
        color: upliftColors,
        line: {{ width: 1.5, color: upliftBorders }},
      }},
      hovertemplate: "%{{customdata}}<extra></extra>",
      customdata: hoverUplift,
      showlegend: false,
    }};

    const layoutUp = {{
      height: 260,
      paper_bgcolor: "{SURFACE}",
      plot_bgcolor: "{SURFACE}",
      font: {{ family: "DM Sans, sans-serif", color: "{TEXT}", size: 12 }},
      margin: {{ t: 12, b: 32, l: SHARED_MARGIN_L, r: SHARED_MARGIN_R }},
      bargap: SHARED_BARGAP,
      annotations: annotationsUp,
      hoverlabel: hlStyle,
      xaxis: {{ ...sharedXAxis }},
      yaxis: {{
        title: {{ text: "<b>Uplift</b>", font: {{ size: 13, color: "{TEXT}", family: "DM Sans" }} }},
        tickformat: ".2%",
        showgrid: true, gridcolor: "{GRID}", gridwidth: 1, griddash: "dot",
        zeroline: true, zerolinecolor: "{BORDER}", zerolinewidth: 2,
        linecolor: "{BORDER}",
        tickfont: {{ size: 11, color: "{TEXT}", family: "DM Sans", weight: "bold" }},
        tickcolor: "{BORDER}",
        automargin: true,
        fixedrange: true,
      }},
    }};

    const elUp = document.getElementById("chart-uplift");
    if (!gdUp) {{
      Plotly.newPlot(elUp, [upliftTrace], layoutUp, configPlotly);
      gdUp = elUp;
    }} else {{
      Plotly.react(gdUp, [upliftTrace], layoutUp, configPlotly);
    }}

    // ── RR CHART (bottom) ────────────────────────────────────────────────────
    const ctrlTrace = {{
      x: bins, y: t.ctrl_rr, type: "bar", name: "Control",
      marker: {{
        color: CTRL_FILL,
        line: {{ width: 1, color: CTRL_LINE }},
      }},
      hovertemplate: "%{{customdata}}<extra></extra>",
      customdata: hoverCtrl, legendgroup: "ctrl",
    }};
    const trtTrace = {{
      x: bins, y: t.trt_rr, type: "bar", name: "Treatment",
      marker: {{
        color: TRT_FILL,
        line: {{ width: 1, color: TRT_LINE }},
      }},
      hovertemplate: "%{{customdata}}<extra></extra>",
      customdata: hoverTrt, legendgroup: "trt",
    }};

    const layoutRr = {{
      height: 300,
      paper_bgcolor: "{SURFACE}",
      plot_bgcolor: "{SURFACE}",
      font: {{ family: "DM Sans, sans-serif", color: "{TEXT}", size: 12 }},
      margin: {{ t: 8, b: 56, l: SHARED_MARGIN_L, r: SHARED_MARGIN_R }},
      barmode: "group",
      bargap: SHARED_BARGAP,
      bargroupgap: 0.08,
      showlegend: true,
      legend: {{
        orientation: "h",
        x: 1, xanchor: "right",
        y: 1.02, yanchor: "bottom",
        bgcolor: "rgba(0,0,0,0)",
        font: {{ size: 12, color: "{TEXT}" }},
      }},
      hoverlabel: hlStyle,
      xaxis: {{
        ...sharedXAxis,
        title: {{ text: "<b>" + D.colAxisLabels[aC] + "</b>", font: {{ size: 13, color: "{TEXT}", family: "DM Sans" }} }},
      }},
      yaxis: {{
        title: {{ text: "<b>Reactivation Rate</b>", font: {{ size: 13, color: "{TEXT}", family: "DM Sans" }} }},
        tickformat: ".1%",
        showgrid: true, gridcolor: "{GRID}", gridwidth: 1, griddash: "dot",
        zeroline: false,
        linecolor: "{BORDER}",
        tickfont: {{ size: 11, color: "{TEXT}", family: "DM Sans", weight: "bold" }},
        tickcolor: "{BORDER}",
        automargin: true,
        fixedrange: true,
      }},
    }};

    const elRr = document.getElementById("chart-rr");
    if (!gdRr) {{
      Plotly.newPlot(elRr, [ctrlTrace, trtTrace], layoutRr, configPlotly);
      gdRr = elRr;
    }} else {{
      Plotly.react(gdRr, [ctrlTrace, trtTrace], layoutRr, configPlotly);
    }}
  }}

  // ── Insight generator ──────────────────────────────────────────────────────
  function updateInsight(ti) {{
    const t = D.tables[ti];
    if (!t) return;

    const el = document.getElementById("insight-text");
    const expLabel = D.expLabels[aE];
    const colLabel = D.colLabels[aC];

    if (t.best_idx != null && t.uplift[t.best_idx] != null) {{
      const bestBin = t.bins[t.best_idx];
      const bestUplift = t.uplift[t.best_idx];
      const bestN = t.trt_n[t.best_idx];
      const valClass = bestUplift >= 0 ? "up-val" : "down-val";

      currentInsightText = `The ${{expLabel}} incentive drives the strongest reactivation among customers with ${{colLabel.toLowerCase()}} around ${{bestBin}} (${{spct(bestUplift)}} uplift, n=${{bestN.toLocaleString()}}).`;

      el.innerHTML = `The <strong>${{expLabel}}</strong> incentive drives the strongest reactivation among customers with <strong>${{colLabel.toLowerCase()}}</strong> around <strong>${{bestBin}}</strong> (<span class="${{valClass}}">${{spct(bestUplift)}}</span> uplift, n=${{bestN.toLocaleString()}}).`;
    }} else {{
      currentInsightText = `No clearly significant uplift detected for ${{expLabel}} across ${{colLabel.toLowerCase()}} segments.`;
      el.innerHTML = currentInsightText;
    }}

    // Reset copy button
    const copyBtn = document.getElementById("copy-btn");
    copyBtn.textContent = "Copy";
    copyBtn.classList.remove("copied");
  }}

  // ── Copy insight ───────────────────────────────────────────────────────────
  window.copyInsight = function() {{
    navigator.clipboard.writeText(currentInsightText).then(() => {{
      const btn = document.getElementById("copy-btn");
      btn.textContent = "Copied ✓";
      btn.classList.add("copied");
      setTimeout(() => {{ btn.textContent = "Copy"; btn.classList.remove("copied"); }}, 2000);
    }});
  }};

  // ── Initial render ─────────────────────────────────────────────────────────
  apply();

}})();
</script>
</body>
</html>"""


# ╔═══════════════════════════════════════════════════════════════════════════════╗
# ║  MAIN                                                                        ║
# ╚═══════════════════════════════════════════════════════════════════════════════╝

if __name__ == "__main__":
    print("Loading data...")
    df = load_data()
    print(f"  {len(df):,} rows, {df['experiment_k'].nunique()} experiments\n")

    print("Building CATE Explorer v2 dashboard...")
    run_cate_explorer_v2(df, renderer="browser")
    print("Done.")

Loading data...
  421,814 rows, 8 experiments

Building CATE Explorer v2 dashboard...
  → Output\cate_explorer_v2.html
Done.
